# Titanic Survival — Sandbox Notebook

**Purpose:** This is the data scientist's *lab bench* — an interactive space to explore data, inspect intermediate pipeline states, and debug module logic.

**Ground rules:**
- This notebook runs **entirely in memory** and writes **no production artifacts** to disk.
- The only authorised entry point for writing `data/processed/clean.csv`, `models/model.joblib`, and `reports/predictions.csv` is `python -m src.main`.
- Keep `PROTECT_TEST_SET = True` while experimenting. Only flip it to `False` when you are ready for a final, one-shot evaluation.

---

In [ ]:
# --- Robust repo root discovery so `import src...` works everywhere ---
import os
import sys
from pathlib import Path

def find_repo_root(start=None):
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and ((p / "config.yaml").exists() or (p / ".git").exists()):
            return p
    return start

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root :", REPO_ROOT)
print("Working dir:", Path.cwd())

## Settings

All configuration is defined here in one place. `FEATURE_COLS` must match the columns produced by `clean_dataframe()` — note that `FamilySize` and `Title` are **engineered** features that do not exist in the raw CSV; they are created during cleaning.

In [ ]:
TARGET       = "Survived"
PROBLEM_TYPE = "classification"
RANDOM_STATE = 42
RAW_DATA_PATH = Path("data/raw/titanic.csv")

# Feature groups — must match the columns produced by clean_dataframe()
NUMERIC_COLS      = ["Age", "Fare", "FamilySize"]          # -> StandardScaler
CATEGORICAL_COLS  = ["Pclass", "Sex", "Embarked", "Title"] # -> OneHotEncoder
QUANTILE_BIN_COLS = []                                      # optional; unused in baseline

FEATURE_COLS  = NUMERIC_COLS + CATEGORICAL_COLS
REQUIRED_COLS = FEATURE_COLS + [TARGET]

TEST_SIZE = 0.15
VAL_SIZE  = 0.15  # fraction of the full dataset, carved from the non-test remainder

# Flip to False only for a final, one-shot test-set evaluation
PROTECT_TEST_SET = True

print("Settings OK.")
print(f"  Numeric features   : {NUMERIC_COLS}")
print(f"  Categorical features: {CATEGORICAL_COLS}")
print(f"  Target             : {TARGET}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report

from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference

print("Imports OK.")

## 1 — Load Raw Data

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
df_raw_snapshot = df_raw.copy(deep=True)  # immutable reference for later comparisons

print(f"Raw data shape: {df_raw.shape}")
df_raw.head(3)

## 2 — Exploratory Data Analysis

This section surfaces the key data quality and distribution facts that drive cleaning and feature engineering decisions downstream.

In [ ]:
# ── Schema and missing values ─────────────────────────────────────────────
print("── dtypes ──")
display(df_raw.dtypes)

print("\n── Missing value rate (sorted) ──")
display(
    df_raw.isna().mean()
    .sort_values(ascending=False)
    .head(15)
    .rename("missing_rate")
)

print(f"\nShape                           : {df_raw.shape}")
print(f"Duplicate rows                  : {df_raw.duplicated().sum()}")
print(f"Target prevalence (Survived=1)  : {df_raw[TARGET].mean():.1%}")

In [ ]:
# ── EDA visualisations ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Titanic EDA — Raw Data", fontsize=14, fontweight="bold")

# 1. Target class balance
ax = axes[0, 0]
df_raw[TARGET].value_counts().sort_index().plot(
    kind="bar", ax=ax, color=["#d9534f", "#5cb85c"], edgecolor="white"
)
ax.set_title("Target Class Balance")
ax.set_xlabel("Survived")
ax.set_ylabel("Count")
ax.set_xticklabels(["Did not survive (0)", "Survived (1)"], rotation=0)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2, p.get_height() + 5),
                ha="center", fontsize=10)

# 2. Survival rate by Sex
ax = axes[0, 1]
df_raw.groupby("Sex")[TARGET].mean().plot(
    kind="bar", ax=ax, color="#5bc0de", edgecolor="white"
)
ax.set_title("Survival Rate by Sex")
ax.set_ylabel("Survival Rate")
ax.set_xticklabels(["Female", "Male"], rotation=0)
ax.set_ylim(0, 1)
ax.axhline(df_raw[TARGET].mean(), color="grey", linestyle="--",
           linewidth=1, label="Overall avg")
ax.legend(fontsize=8)

# 3. Survival rate by Pclass
ax = axes[0, 2]
df_raw.groupby("Pclass")[TARGET].mean().plot(
    kind="bar", ax=ax, color="#f0ad4e", edgecolor="white"
)
ax.set_title("Survival Rate by Pclass")
ax.set_ylabel("Survival Rate")
ax.set_xticklabels(["1st", "2nd", "3rd"], rotation=0)
ax.set_ylim(0, 1)

# 4. Age distribution by survival
ax = axes[1, 0]
df_raw[df_raw[TARGET] == 0]["Age"].dropna().hist(
    ax=ax, bins=25, alpha=0.6, color="#d9534f", label="Did not survive"
)
df_raw[df_raw[TARGET] == 1]["Age"].dropna().hist(
    ax=ax, bins=25, alpha=0.6, color="#5cb85c", label="Survived"
)
ax.set_title("Age Distribution by Survival")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

# 5. Fare distribution by survival (log scale)
ax = axes[1, 1]
df_raw[df_raw[TARGET] == 0]["Fare"].dropna().hist(
    ax=ax, bins=30, alpha=0.6, color="#d9534f", label="Did not survive"
)
df_raw[df_raw[TARGET] == 1]["Fare"].dropna().hist(
    ax=ax, bins=30, alpha=0.6, color="#5cb85c", label="Survived"
)
ax.set_title("Fare Distribution by Survival")
ax.set_xlabel("Fare (£)")
ax.set_ylabel("Count")
ax.set_yscale("log")
ax.legend(fontsize=8)

# 6. Survival rate by Embarked
ax = axes[1, 2]
df_raw.groupby("Embarked")[TARGET].mean().sort_values(ascending=False).plot(
    kind="bar", ax=ax, color="#9b59b6", edgecolor="white"
)
ax.set_title("Survival Rate by Embarkation Port")
ax.set_ylabel("Survival Rate")
ax.set_xticklabels(
    ["C (Cherbourg)", "Q (Queenstown)", "S (Southampton)"],
    rotation=20, ha="right"
)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 3 — Clean Data

After calling `clean_dataframe()` we verify that:
1. All missing values have been imputed (Age, Fare, Embarked).
2. The two engineered features `FamilySize` and `Title` are present with sensible distributions.

In [ ]:
df_clean = clean_dataframe(df_raw, target_column=TARGET)

print(f"Shape before cleaning : {df_raw_snapshot.shape}")
print(f"Shape after  cleaning : {df_clean.shape}")
df_clean.head(3)

In [ ]:
# ── Post-cleaning verification ────────────────────────────────────────────
print("── Remaining nulls after cleaning ──")
null_counts = df_clean.isna().sum()
if null_counts.sum() == 0:
    print("  None — all columns fully imputed.")
else:
    print(null_counts[null_counts > 0])

print("\n── Engineered feature: FamilySize (value counts) ──")
assert "FamilySize" in df_clean.columns, "ERROR: FamilySize missing after cleaning!"
print(df_clean["FamilySize"].value_counts().sort_index())

print("\n── Engineered feature: Title (value counts) ──")
assert "Title" in df_clean.columns, "ERROR: Title missing after cleaning!"
print(df_clean["Title"].value_counts())

print("\n── Final columns ──")
print(list(df_clean.columns))

## 4 — Validate

`validate_dataframe()` is a **fail-fast gate**. If it raises a `ValueError` here, stop and fix the cleaning step — do not attempt to train on invalid data.

In [ ]:
validate_dataframe(df_clean, required_columns=REQUIRED_COLS)
print("Validation passed — data is safe to split and train on.")

## 5 — Train / Val / Test Split

**Leakage prevention:** the split happens here, *before* the feature preprocessor is built or fitted. The `ColumnTransformer` will only ever see `X_train` during `.fit()`. The val and test sets are transformed later using statistics computed exclusively from training data.

In [ ]:
X = df_clean[FEATURE_COLS].copy()
y = df_clean[TARGET].copy()

# Step 1: carve out the held-out test set
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Step 2: split the remainder into train and val
val_fraction_of_remaining = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=val_fraction_of_remaining,
    random_state=RANDOM_STATE,
    stratify=y_tmp,
)

print(f"Train : {X_train.shape}  |  survival rate: {y_train.mean():.1%}")
print(f"Val   : {X_val.shape}   |  survival rate: {y_val.mean():.1%}")
print(f"Test  : {X_test.shape}  |  survival rate: {y_test.mean():.1%}")
print("\nStratified split confirmed — class proportions are consistent across all splits.")

## 6 — Build Feature Preprocessor

`get_feature_preprocessor()` returns an **unfitted** `ColumnTransformer`. Nothing is learned from data in this cell. Fitting happens inside `train_model()` wrapped in a sklearn `Pipeline`, ensuring no information from val or test sets leaks into the preprocessing step.

In [ ]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=QUANTILE_BIN_COLS,
    categorical_onehot_cols=CATEGORICAL_COLS,
    numeric_passthrough_cols=NUMERIC_COLS,
    n_bins=3,
)

preprocessor  # displays the unfitted sklearn diagram

## 7 — Train Model

In [ ]:
model = train_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type=PROBLEM_TYPE,
)

model  # displays the fitted Pipeline diagram

## 8 — Evaluate on Validation Set

`evaluate_model()` returns a single scalar metric (weighted F1) and prints the full classification report to the console. The cells below render the confusion matrix and ROC curve visually for deeper diagnostic inspection.

In [ ]:
val_f1 = evaluate_model(model, X_val, y_val, problem_type=PROBLEM_TYPE)
print(f"\nValidation weighted F1: {val_f1:.4f}")

In [ ]:
# ── Visual diagnostics ───────────────────────────────────────────────────
y_val_pred  = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Validation Set Diagnostics", fontsize=13, fontweight="bold")

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_pred,
    display_labels=["Did not survive", "Survived"],
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix — Validation Set")

# ROC curve
RocCurveDisplay.from_predictions(
    y_val, y_val_proba,
    name="Logistic Regression",
    ax=axes[1],
)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random baseline")
axes[1].set_title("ROC Curve — Validation Set")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n── Full Classification Report ──")
print(classification_report(
    y_val, y_val_pred,
    target_names=["Did not survive", "Survived"]
))

## 9 — Final Evaluation on Test Set (Guarded)

> ⚠️ **Why the guard exists:** The test set is your single, unbiased estimate of real-world performance. Every time you examine test metrics and then adjust your model, you are implicitly overfitting to the test set — even if you do not retrain on it directly. For that reason `PROTECT_TEST_SET = True` by default. Only flip it to `False` when you have made your **final** modelling decision and need one conclusive number to report.

In [ ]:
if not PROTECT_TEST_SET:
    print("── FINAL TEST SET EVALUATION ──")
    test_f1 = evaluate_model(model, X_test, y_test, problem_type=PROBLEM_TYPE)
    print(f"\nTest weighted F1: {test_f1:.4f}")
else:
    print("Test set is protected.")
    print("Set PROTECT_TEST_SET = False in the Settings cell to run the final evaluation.")

## 10 — Inference

`run_inference()` returns a **four-column** DataFrame: `prediction`, `survival_probability`, `outcome`, and `high_confidence`. We use the first 10 rows of the validation set as a stand-in for genuinely unseen passengers.

In [ ]:
# Simulate new, unseen passengers using the first 10 rows of the validation set
X_infer = X_val.head(10).copy()

preds = run_inference(model, X_infer)

print("\n── Enriched Predictions (all 4 contract columns) ──")
display(preds)

print(f"\nPredicted survival rate    : {preds['prediction'].mean():.1%}")
print(f"Mean survival probability  : {preds['survival_probability'].mean():.3f}")
print(f"High-confidence predictions: {preds['high_confidence'].sum()} / {len(preds)}")

In [ ]:
# ── Explore the enriched output ───────────────────────────────────────────

# Borderline passengers — model is uncertain (probability between 0.3 and 0.7)
borderline = preds[~preds["high_confidence"]]
print(f"Borderline predictions (low confidence): {len(borderline)} row(s)")
if len(borderline) > 0:
    display(borderline)

# Probability bar chart
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(
    range(len(preds)),
    preds["survival_probability"],
    color=["#5cb85c" if p == 1 else "#d9534f" for p in preds["prediction"]],
    edgecolor="white",
)
ax.axhline(0.7, color="grey", linestyle="--", linewidth=1, label="High-confidence threshold (0.7)")
ax.axhline(0.3, color="grey", linestyle=":",  linewidth=1, label="Low-confidence threshold  (0.3)")
ax.set_title("Survival Probability — Sample Inference (green=Survived, red=Did not survive)")
ax.set_xlabel("Passenger (sample position)")
ax.set_ylabel("P(Survived)")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Cross-reference against actual labels
preds_vs_actual = preds.copy()
preds_vs_actual["actual"] = y_val.head(10).values
preds_vs_actual["correct"] = preds_vs_actual["prediction"] == preds_vs_actual["actual"]
print(f"\nAccuracy on these 10 rows: {preds_vs_actual['correct'].mean():.0%}")
display(
    preds_vs_actual[[
        "prediction", "actual", "survival_probability",
        "outcome", "high_confidence", "correct"
    ]]
)

## 11 — Run the Orchestrator

The cells above run the pipeline **in memory** and write nothing to disk. To produce the canonical production artifacts, run from your terminal:

```bash
python -m src.main
```

The cell below invokes the orchestrator via `subprocess` as a smoke test. It prints stdout/stderr and explains common failure causes rather than raising a bare `RuntimeError`.

In [ ]:
import subprocess

cmd = [sys.executable, "-m", "src.main"]
print("Running:", " ".join(cmd))

r = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True)

print("---- STDOUT ----")
print(r.stdout)
print("---- STDERR ----")
print(r.stderr)

if r.returncode != 0:
    print(f"\n--- Orchestrator exited with code {r.returncode} ---")
    print("Common causes:")
    print("  1. SETTINGS['raw_data_path'] in main.py points to a file that does not exist.")
    print("  2. SETTINGS['is_example_config'] is True but the expected dummy CSV is missing.")
    print("  3. A bug was introduced in one of the student code sections.")
    print("\nFix the issue and re-run this cell.")
else:
    print("\nOrchestrator completed successfully. Artifacts written to disk.")